# Exciter Models - Steady-State Test (SP domain)

Attaches each exciter to a Kundur `SynchronGenerator4OrderVBR` (SP domain) and checks that $E_f$ and $V_t$ are at steady state.

In [ ]:
import math
import numpy as np
import dpsimpy

# Kundur machine parameters
# P. Kundur, "Power System Stability and Control", Example 3.2, pp. 102
NOM_POWER = 555e6
NOM_VOLTAGE = 24e3  # phase-to-phase RMS, V
NOM_FREQ = 60.0
H = 3.7
Ld = 1.8099
Lq = 1.7600
Ll = 0.15
Ld_t = 0.2999
Lq_t = 0.6500
Td0_t = 8.0669
Tq0_t = 0.9991

# SMIB operating point (ScenarioConfig3 from dpsim/examples/cxx/Examples.h)
INIT_ACTIVE_POWER = 300e6
INIT_VOLT_ANGLE = -math.pi / 2
INIT_VOLT = complex(
    NOM_VOLTAGE * math.cos(INIT_VOLT_ANGLE),
    NOM_VOLTAGE * math.sin(INIT_VOLT_ANGLE),
)

## ExciterDC1Simp

IEEE DC1A without lead-lag ($T_b$, $T_c$ neglected). Parameters: Eremia (2013), pp. 96, 106.

In [ ]:
sim_name = "SP_SynGen4OrderVBR_ExciterDC1Simp"
dpsimpy.Logger.set_log_dir("logs/" + sim_name)

n1 = dpsimpy.sp.SimNode("n1", dpsimpy.PhaseType.Single, [INIT_VOLT])

gen = dpsimpy.sp.ph1.SynchronGenerator4OrderVBR("SynGen")
gen.set_operational_parameters_per_unit(
    NOM_POWER,
    NOM_VOLTAGE,
    NOM_FREQ,
    H,
    Ld,
    Lq,
    Ll,
    Ld_t,
    Lq_t,
    Td0_t,
    Tq0_t,
)
gen.set_initial_values(complex(INIT_ACTIVE_POWER, 0), INIT_ACTIVE_POWER, INIT_VOLT)
gen.set_model_as_norton_source(True)
gen.connect([n1])

exciter = dpsimpy.signal.ExciterDC1Simp("Exciter")
p = dpsimpy.signal.ExciterDC1SimpParameters()
p.Ka = 46.0
p.Ta = 0.06
p.Kef = -0.0435
p.Tef = 0.46
p.Kf = 0.1
p.Tf = 1.0
p.Tr = 0.02
p.Aef = 0.33
p.Bef = 0.1
p.MaxVa = 1.0
p.MinVa = -0.9
exciter.set_parameters(p)
gen.add_exciter(exciter)

load = dpsimpy.sp.ph1.Load("Load")
load.set_parameters(INIT_ACTIVE_POWER, 0, NOM_VOLTAGE)
load.connect([n1])

system = dpsimpy.SystemTopology(NOM_FREQ, [n1], [gen, load])

sim = dpsimpy.Simulation(sim_name)
sim.set_system(system)
sim.set_time_step(1e-3)
sim.set_final_time(2.0)
sim.set_domain(dpsimpy.Domain.SP)
sim.do_init_from_nodes_and_terminals(True)
sim.run()

Ef = gen.attr("Ef").get()
Vt_pu = abs(gen.attr("v_intf").get()[0, 0]) / NOM_VOLTAGE
print(f"ExciterDC1Simp: Ef = {Ef:.4f} pu,  Vt = {Vt_pu:.4f} pu")

assert math.isfinite(Ef), f"Ef is not finite: {Ef}"
assert Ef != 0.0, "Ef is zero at end of simulation"
assert (
    0.95 <= Vt_pu <= 1.05
), f"Terminal voltage {Vt_pu:.4f} pu outside \u00b15\u0020% of 1.0 pu"

## ExciterDC1

IEEE DC1A with lead-lag block $(1+sT_c)/(1+sT_b)$. Same Eremia parameters as DC1Simp; $T_c=0.1\,\text{s}$, $T_b=1.0\,\text{s}$ chosen to satisfy the non-zero divisor constraint.

In [ ]:
sim_name = "SP_SynGen4OrderVBR_ExciterDC1"
dpsimpy.Logger.set_log_dir("logs/" + sim_name)

n1 = dpsimpy.sp.SimNode("n1", dpsimpy.PhaseType.Single, [INIT_VOLT])

gen = dpsimpy.sp.ph1.SynchronGenerator4OrderVBR("SynGen")
gen.set_operational_parameters_per_unit(
    NOM_POWER,
    NOM_VOLTAGE,
    NOM_FREQ,
    H,
    Ld,
    Lq,
    Ll,
    Ld_t,
    Lq_t,
    Td0_t,
    Tq0_t,
)
gen.set_initial_values(complex(INIT_ACTIVE_POWER, 0), INIT_ACTIVE_POWER, INIT_VOLT)
gen.set_model_as_norton_source(True)
gen.connect([n1])

exciter = dpsimpy.signal.ExciterDC1("Exciter")
p = dpsimpy.signal.ExciterDC1Parameters()
p.Ka = 46.0
p.Ta = 0.06
p.Tb = 1.0
p.Tc = 0.1
p.Kef = -0.0435
p.Tef = 0.46
p.Kf = 0.1
p.Tf = 1.0
p.Tr = 0.02
p.Aef = 0.33
p.Bef = 0.1
p.MaxVa = 1.0
p.MinVa = -0.9
exciter.set_parameters(p)
gen.add_exciter(exciter)

load = dpsimpy.sp.ph1.Load("Load")
load.set_parameters(INIT_ACTIVE_POWER, 0, NOM_VOLTAGE)
load.connect([n1])

system = dpsimpy.SystemTopology(NOM_FREQ, [n1], [gen, load])

sim = dpsimpy.Simulation(sim_name)
sim.set_system(system)
sim.set_time_step(1e-3)
sim.set_final_time(2.0)
sim.set_domain(dpsimpy.Domain.SP)
sim.do_init_from_nodes_and_terminals(True)
sim.run()

Ef = gen.attr("Ef").get()
Vt_pu = abs(gen.attr("v_intf").get()[0, 0]) / NOM_VOLTAGE
print(f"ExciterDC1: Ef = {Ef:.4f} pu,  Vt = {Vt_pu:.4f} pu")

assert math.isfinite(Ef), f"Ef is not finite: {Ef}"
assert Ef != 0.0, "Ef is zero at end of simulation"
assert (
    0.95 <= Vt_pu <= 1.05
), f"Terminal voltage {Vt_pu:.4f} pu outside \u00b15\u0020% of 1.0 pu"

## ExciterST1Simp

Pure proportional gain: $E_f = K_a (V_r - V_{\text{ref}})$. $K_a=46$ reused from Eremia DC1Simp for consistency (no dedicated source). $V_{\text{ref}}$ is back-calculated at init, so the steady-state point is independent of $K_a$.

In [ ]:
sim_name = "SP_SynGen4OrderVBR_ExciterST1Simp"
dpsimpy.Logger.set_log_dir("logs/" + sim_name)

n1 = dpsimpy.sp.SimNode("n1", dpsimpy.PhaseType.Single, [INIT_VOLT])

gen = dpsimpy.sp.ph1.SynchronGenerator4OrderVBR("SynGen")
gen.set_operational_parameters_per_unit(
    NOM_POWER,
    NOM_VOLTAGE,
    NOM_FREQ,
    H,
    Ld,
    Lq,
    Ll,
    Ld_t,
    Lq_t,
    Td0_t,
    Tq0_t,
)
gen.set_initial_values(complex(INIT_ACTIVE_POWER, 0), INIT_ACTIVE_POWER, INIT_VOLT)
gen.set_model_as_norton_source(True)
gen.connect([n1])

exciter = dpsimpy.signal.ExciterST1Simp("Exciter")
p = dpsimpy.signal.ExciterST1Parameters()
p.Ka = 46.0
p.Tr = 0.02
p.MaxVa = 7.0
p.MinVa = -7.0
exciter.set_parameters(p)
gen.add_exciter(exciter)

load = dpsimpy.sp.ph1.Load("Load")
load.set_parameters(INIT_ACTIVE_POWER, 0, NOM_VOLTAGE)
load.connect([n1])

system = dpsimpy.SystemTopology(NOM_FREQ, [n1], [gen, load])

sim = dpsimpy.Simulation(sim_name)
sim.set_system(system)
sim.set_time_step(1e-3)
sim.set_final_time(2.0)
sim.set_domain(dpsimpy.Domain.SP)
sim.do_init_from_nodes_and_terminals(True)
sim.run()

Ef = gen.attr("Ef").get()
Vt_pu = abs(gen.attr("v_intf").get()[0, 0]) / NOM_VOLTAGE
print(f"ExciterST1Simp: Ef = {Ef:.4f} pu,  Vt = {Vt_pu:.4f} pu")

assert math.isfinite(Ef), f"Ef is not finite: {Ef}"
assert Ef != 0.0, "Ef is zero at end of simulation"
assert (
    0.95 <= Vt_pu <= 1.05
), f"Terminal voltage {Vt_pu:.4f} pu outside \u00b15\u0020% of 1.0 pu"

## ExciterStatic

Lead-lag block with anti-windup. $K_a=46$ from Eremia. Ref.: Roehder et al., *Transmission system stability assessment within an integrated grid development process*.

In [ ]:
sim_name = "SP_SynGen4OrderVBR_ExciterStatic"
dpsimpy.Logger.set_log_dir("logs/" + sim_name)

n1 = dpsimpy.sp.SimNode("n1", dpsimpy.PhaseType.Single, [INIT_VOLT])

gen = dpsimpy.sp.ph1.SynchronGenerator4OrderVBR("SynGen")
gen.set_operational_parameters_per_unit(
    NOM_POWER,
    NOM_VOLTAGE,
    NOM_FREQ,
    H,
    Ld,
    Lq,
    Ll,
    Ld_t,
    Lq_t,
    Td0_t,
    Tq0_t,
)
gen.set_initial_values(complex(INIT_ACTIVE_POWER, 0), INIT_ACTIVE_POWER, INIT_VOLT)
gen.set_model_as_norton_source(True)
gen.connect([n1])

exciter = dpsimpy.signal.ExciterStatic("Exciter")
p = dpsimpy.signal.ExciterStaticParameters()
p.Ka = 46.0
p.Ta = 0.1
p.Tb = 0.5
p.Te = 0.5
p.Tr = 0.02
p.MaxEfd = 10.0
p.MinEfd = -10.0
p.Kbc = 0.0
exciter.set_parameters(p)
gen.add_exciter(exciter)

load = dpsimpy.sp.ph1.Load("Load")
load.set_parameters(INIT_ACTIVE_POWER, 0, NOM_VOLTAGE)
load.connect([n1])

system = dpsimpy.SystemTopology(NOM_FREQ, [n1], [gen, load])

sim = dpsimpy.Simulation(sim_name)
sim.set_system(system)
sim.set_time_step(1e-3)
sim.set_final_time(2.0)
sim.set_domain(dpsimpy.Domain.SP)
sim.do_init_from_nodes_and_terminals(True)
sim.run()

Ef = gen.attr("Ef").get()
Vt_pu = abs(gen.attr("v_intf").get()[0, 0]) / NOM_VOLTAGE
print(f"ExciterStatic: Ef = {Ef:.4f} pu,  Vt = {Vt_pu:.4f} pu")

assert math.isfinite(Ef), f"Ef is not finite: {Ef}"
assert Ef != 0.0, "Ef is zero at end of simulation"
assert (
    0.95 <= Vt_pu <= 1.05
), f"Terminal voltage {Vt_pu:.4f} pu outside \u00b15\u0020% of 1.0 pu"